In [3]:
#Import packages for calculation
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pynucastro as pyna

In [ ]:
class MainphaseStar:
    #Stellar evolution simulator that opperates on the core conceit that the star remains in the Main Phase (where the primary energy in the star comes from hydrogen fusion) and remain in thermal and hydrostatic equilibrium


    def __init__(self, temp, dens, pres, pcenthydrogen, radius):
        #Takes 4 1d-arrays of equal size and 1 float.64
        self.t = temp #in kelvin
        self.d = dens #in gram/meters**3
        self.p = pres 
        self.pcenthydrogen = pcenthydrogen #percent by mass of hydrogen for each layer
        self.r = radius #in meters
        #obtains the number of cells as well as the size of each cell 
        self.n = len(temp)
        self.rinc = self.r / self.n
        
        #use pynucastro to load reaction rates for p-p chain and c-n-o chain reactions, assuming the rates are bounded by the slowest reaction rate 
        rl = pyna.ReacLibLibrary()
        self.ppchainrate = rl.get_rate_by_name("p(p,)d")
        self.cnochainrate = rl.get_rate_by_name("n14(p,g)o15")

        #radiation constant as defined by Stefan Bolzman Law
        self.a = 7.5657 * 10 **(-16)
        #units j (m**-3) (K**-4)

        #gravitational constant
        self.G = 6.6743*10**-13
        #units m**3 /(g*s**2)

    def rincrinitialize(self):
        #reinitializes the calculation for the radius increment of each cell
        self.rinc = self.r / self.n
        self.rbycell = (self.n - np.asarray(range(0, self.n)))*self.rinc

    def massbyrad(self):
        #initializes mass array for each cell, where mass is the volume * the density
        self.mass = np.zeros_like(self.t) #Mass of interior of star from each cell
        self.layermass = np.zeros_like(self.t) #mass of material in the layer composed by each cell
        for i in range(1, self.n):
            self.mass[-i] = 4 / 3 * np.pi * (self.rinc * i) ** 3 * (np.sum(self.d[-i:])/i)  
        for i in range(1, self.n):
            self.layermass[-i] = self.mass[-i] - np.sum(self.mass[-i-1:])

    def fusionenergy(self, density, perhydro, temp):
        #calculate fusion energy per unit mass based on temp and density of hydrogen- using p-p and c-n-o fusion paths 
        #Note: pynucastro assumes rho density is in grams/cm**3 thus we have to convert   
        
        #evaluates reaction rates per cm^3 
        ppeval = self.ppchainrate.eval(temp, density*perhydro/(10**6))
        cnoeval = self.cnochainrate.eval(temp, density*perhydro/(10**6))

        #convert to reaction rates per unit mass
        ppeval = ppeval*10**6/density
        cnoeval = cnoeval*10**6/density

        #evaluate energy result per unit mass then converts mega electron volt to electron volt
        energyperunitmass = (ppeval*26.72 + cnoeval*25)*1000000 
        return energyperunitmass
    
    def drdm(self, m, r, rho):   
        #mass conservation equation
        return 1/(4*np.pi*r**2 *rho)

    def dpdm(self, m, p, r):
        #hydrostatic equilibrium
        dpdm = -G*m/(4*np.pi**4*r)
    
    def dLdm(self, m, L, nuc, r1, r2, rho):
        grav = 16/5*np.pi*self.G*rho**2*(r2**5 - r1**5)
        return nuc + grav 

    def massolve(self, masses):
        #Takes list of t masses to solve system at. First mass is assumed to be mass of original system 
        #returns a t by n array for each system quantity of values at each radius interval n for that mass
        #evaluates from center out for each time
        
        #initialize arrays
        self.massbyrad()
        self.rincrinitialize()
        temp = np.zeros((len(masses), self.n)) 
        dens = np.zeros((len(masses), self.n)) 
        pres = np.zeros((len(masses), self.n)) 
        radis = np.zeros((len(masses), self.n)) 
        pcent = np.zeros((len(masses), self.n)) 
        efromrad = np.zeros((len(masses), self.n)) 
        intmass = np.zeros((len(masses), self.n))
        laymass = np.zeros((len(masses), self.n))
        temp[0, :] = self.t
        dens[0, :] = self.d
        pres[0, :] = self.p
        radis[0, :] = self.rbycell
        pcent[0, :] = self.pcenthydrogen
        intmass[0, :] = self.mass
        laymass[0, :] = self.layermass

        for q in range(1, len(masses)):

            #calculate the mass difference between this stage and previous mass
            massdif = masses[q] - masses[q-1]

            #evaluate per unit fusion energy at each cell and convert that into difference in mass across layers
            for i in range(0, self.n):
                efromrad[q-1, -i] = self.fusionenergy( dens[q-1, -i], pcent[q-1, -i], temp[q-1, -i])
            pcentfusion = efromrad[q-1, :] / np.sum(efromrad[q-1, :])
            massdifbycell = massdif * pcentfusion
            laymass[q, :] = laymass[q-1, :] - massdifbycell

            #update hydrogen percentages
            pcent[q, :] = (pcent[q-1, :] * laymass[q-1, :] - massdifbycell) / laymass[q-1, :]

            #solve for radiuses
            for i in range(0, self.n):
                rho = sum(dens[q-1, i:])/(self.n - i)
                output = sp.integrate.solve_ivp(self.drdm, range(masses[q-1], masses[q]), radis[q-1, i], args = (rho))
                radis[q, i] = output[1][-1]

            #solve for preassures 
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.dpdm, range(masses[q-1], masses[q]), pres[q-1, i], args = (radis[q, i]))
                pres[q, i] = output[1][-1]
            


        return 